# Lesson 4.2: StateGraph and reducers vs your dict

**Goal:** run the same six triage nodes from lesson 4.1 inside LangGraph's `StateGraph`, and see that a reducer is your `merge`.

**What you will be able to do**
- Write a state schema as a `TypedDict`, with a reducer on the keys that need one.
- Build a `StateGraph`: `add_node`, `add_edge`, `add_conditional_edges`, `compile`, then `invoke` / `stream`.
- Prove the graph gives the same final state as your hand runner on all 60 incidents.
- Predict what happens without a reducer, and how `add_messages` appends or replaces.

In [1]:
# Setup: find the course repo's nightwatch/ folder (relative to this notebook) and load .env if present.
import os, sys
from pathlib import Path

for up in ("../../nightwatch", "../../../nightwatch", "../nightwatch", "nightwatch"):
    if Path(up, "brightcart").is_dir():
        NW = Path(up)
        break
else:
    raise FileNotFoundError("Open this notebook from <lesson>/notebook inside the course repo: it needs the nightwatch/ folder.")
sys.path.insert(0, str(NW.resolve()))

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))      # walks up from here and loads the repo's .env
except ImportError:
    pass
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
print("nightwatch folder:", NW.as_posix(), "| GROQ_API_KEY:", "found" if HAS_KEY else "not set")
print("This notebook makes no model calls, so no key is needed.")

nightwatch folder: ../../../nightwatch | GROQ_API_KEY: found
This notebook makes no model calls, so no key is needed.


## 1. Recall: how an update joins the state (lesson 4.1)

A node returns only the keys it changed. Your runner folded that update in with `merge`: **notes appended, every other
key overwritten**. Below are the same six nodes, the two decisions and the hand runner from lesson 4.1, unchanged.

In [2]:
import json
from brightcart.incidents import fresh, generate
from brightcart.sim import World

INCIDENTS = {x.id: x for x in generate()}
SIGNS = [("401 Unauthorized", "expired_secret"), ("OOMKilled", "memory_leak"), ("No space left", "disk_full"),
         ("guarded by flag", "bad_flag"), ("db connection", "bad_config"), ("shedding load", "traffic_spike"),
         ("Error:", "bad_deploy")]

def world(state):
    return World(fresh(INCIDENTS[state["incident"]]))

def read_alert(state):
    first = state["alert"].split()[0]
    return {"suspect": first, "notes": ["alert names " + first]}

def check_health(state):
    h = world(state).health()
    return {"unhealthy": h["unhealthy_services"], "notes": [f"unhealthy: {h['unhealthy_services']}"]}

def read_logs(state):
    lines = [x for x in world(state).logs(state["suspect"], 20)
             if ("ERROR" in x or "WARN" in x) and "GC pause" not in x]
    return {"evidence": lines[-2:], "hops": state["hops"] + 1,
            "notes": [f"read {state['suspect']} logs: {len(lines)} bad lines"]}

def follow_upstream(state):
    line = next(e for e in state["evidence"] if "upstream" in e).split()
    callee = line[line.index("upstream") + 1]
    return {"suspect": callee, "notes": [f"errors come from upstream {callee}"]}

def classify(state):
    text = " ".join(state["evidence"])
    cause = next((c for sign, c in SIGNS if sign in text), "unknown")
    return {"cause": cause, "notes": [f"cause = {cause}"]}

def false_alarm(state):
    return {"cause": "false_alarm", "notes": ["all green: false alarm, change nothing"]}

def after_health(state):
    return "read_logs" if state["unhealthy"] else "false_alarm"

def after_logs(state):
    upstream = any("upstream" in e for e in state["evidence"])
    return "follow_upstream" if upstream and state["hops"] < 3 else "classify"

NODES = {f.__name__: f for f in (read_alert, check_health, read_logs, follow_upstream, classify, false_alarm)}

def start_state(incident_id):
    inc = INCIDENTS[incident_id]
    return {"incident": incident_id, "alert": inc.alert, "suspect": None, "unhealthy": [], "evidence": [],
            "hops": 0, "cause": None, "notes": []}

# ---- the hand runner of lesson 4.1 (printing left out)
HAND_END = "END"
EDGES = {"read_alert": "check_health", "check_health": after_health, "read_logs": after_logs,
         "follow_upstream": "read_logs", "classify": HAND_END, "false_alarm": HAND_END}

def merge(state, update):
    new = dict(state)
    for k, v in update.items():
        new[k] = state[k] + v if k == "notes" else v
    return new

def run(state, entry="read_alert", max_steps=10):
    node = entry
    for _ in range(max_steps):
        state = merge(state, NODES[node](state))
        nxt = EDGES[node]
        node = nxt(state) if callable(nxt) else nxt
        if node == HAND_END:
            return state
    raise RuntimeError("step limit")

print(merge({"notes": ["a"], "hops": 2}, {"notes": ["b"], "hops": 1}))

{'notes': ['a', 'b'], 'hops': 1}


## 2. Key words

| Word | Meaning here |
|---|---|
| **StateGraph** | LangGraph's graph builder: add nodes and edges, then `compile()` it |
| **state schema** | a `TypedDict`: a dict type with fixed keys, each with a type |
| **reducer** | a function `(old, update) -> new` that decides how an update joins one key |
| **stream** | run the graph and get each node's update as it happens (`stream_mode="updates"`) |
| **add_messages** | a ready-made reducer for chat messages: new id appended, known id replaced |

## 3. The idea: five pieces, five twins

| Your runner (4.1) | LangGraph |
|---|---|
| the state dict | `class Triage(TypedDict)` |
| `NODES` dict | `g.add_node(name, fn)` |
| `EDGES` dict | `g.add_edge(a, b)` and `g.add_conditional_edges(a, fn, [targets])` |
| `merge` | a reducer next to its key: `notes: Annotated[list, operator.add]` |
| `run(state)` | `g.compile().invoke(state)` |

A reducer by hand: `operator.add(old, new)` on two lists glues them, exactly `state[k] + v` in your merge. No reducer means
the update replaces the old value: the `else` branch of merge.

**Hand trace of `upstream_deploy-0`**, four keys:

| step | node | suspect | hops | evidence | notes |
|---|---|---|---|---|---|
| 1 | read_alert | checkout | 0 | [] | 1 |
| 2 | check_health | checkout | 0 | [] | 2 |
| 3 | read_logs | checkout | 1 | 2 checkout lines ("upstream payments returned 500") | 3 |
| 4 | follow_upstream | payments | 1 | same 2 lines | 4 |
| 5 | read_logs | payments | 2 | **1** line (KeyError), replacing the 2 old ones | 5 |
| 6 | classify | payments | 2 | 1 line | 6 |

Notes grew to 6 thanks to its reducer; evidence was overwritten: your merge's two rules.

In [3]:
import operator
old, new = ["alert names checkout"], ["unhealthy: ['checkout', 'payments']"]
print("operator.add:", operator.add(old, new))
print("merge line  :", old + new)
print("no reducer  :", new)

operator.add: ['alert names checkout', "unhealthy: ['checkout', 'payments']"]
merge line  : ['alert names checkout', "unhealthy: ['checkout', 'payments']"]
no reducer  : ["unhealthy: ['checkout', 'payments']"]


## 4. Write it yourself: the runner you already have
Your hand runner on `upstream_deploy-0` gives the final state we must match:

In [4]:
hand = run(start_state("upstream_deploy-0"))
print("suspect", hand["suspect"], "| hops", hand["hops"], "| cause", hand["cause"])
print("evidence", hand["evidence"])
print(len(hand["notes"]), "notes")

suspect payments | hops 2 | cause bad_deploy
evidence ["04:13 ERROR payments KeyError: 'currency' in serializer (v1.3.5)"]
6 notes


## 5. The library version: the same flow in StateGraph
### 5a. The schema with a reducer

In [5]:
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from importlib.metadata import version
print("langgraph", version("langgraph"))

class Triage(TypedDict):
    incident: str
    alert: str
    suspect: str | None
    unhealthy: list
    evidence: list
    hops: int
    cause: str | None
    notes: Annotated[list, operator.add]        # reducer: new notes are APPENDED

class TriageNoReducer(TypedDict):               # same keys, no reducer on notes
    incident: str
    alert: str
    suspect: str | None
    unhealthy: list
    evidence: list
    hops: int
    cause: str | None
    notes: list                                 # default: the last write wins

langgraph 1.2.22


### 5b. Nodes, edges, compile
Each line next to the `EDGES` entry it replaces. The list in `add_conditional_edges` names the nodes the decision may return, so LangGraph can check and draw the graph before it runs.

In [6]:
def build(schema):
    g = StateGraph(schema)
    for name, fn in NODES.items():                 # NODES dict -> add_node
        g.add_node(name, fn)
    g.add_edge(START, "read_alert")                # entry="read_alert"
    g.add_edge("read_alert", "check_health")       # "read_alert": "check_health"
    g.add_conditional_edges("check_health", after_health, ["read_logs", "false_alarm"])
    g.add_conditional_edges("read_logs", after_logs, ["follow_upstream", "classify"])
    g.add_edge("follow_upstream", "read_logs")     # the cycle
    g.add_edge("classify", END)
    g.add_edge("false_alarm", END)
    return g.compile()

graph = build(Triage)
print(type(graph).__name__, "| nodes:", [n for n in graph.get_graph().nodes])

CompiledStateGraph | nodes: ['__start__', 'read_alert', 'check_health', 'read_logs', 'follow_upstream', 'classify', 'false_alarm', '__end__']


### 5c. invoke and stream

In [7]:
for i, chunk in enumerate(graph.stream(start_state("upstream_deploy-0"), stream_mode="updates"), 1):
    (node, update), = chunk.items()
    print(f"step {i} {node:16} update {json.dumps({k: v for k, v in update.items() if k != 'notes'})}")
final_graph = graph.invoke(start_state("upstream_deploy-0"))
print("final cause", final_graph["cause"], "| notes:", len(final_graph["notes"]))

step 1 read_alert       update {"suspect": "checkout"}
step 2 check_health     update {"unhealthy": ["checkout", "payments"]}
step 3 read_logs        update {"evidence": ["04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge", "04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge"], "hops": 1}
step 4 follow_upstream  update {"suspect": "payments"}
step 5 read_logs        update {"evidence": ["04:13 ERROR payments KeyError: 'currency' in serializer (v1.3.5)"], "hops": 2}
step 6 classify         update {"cause": "bad_deploy"}
final cause bad_deploy | notes: 6


### 5d. Check: is it really the same?
Run both runners on every incident and compare the final dicts.

In [8]:
print("upstream_deploy-0:", run(start_state("upstream_deploy-0")) == final_graph)
same = sum(run(start_state(i)) == graph.invoke(start_state(i)) for i in INCIDENTS)
print(f"all 60 incidents: {same}/60 final states identical")

upstream_deploy-0: True


all 60 incidents: 60/60 final states identical


Identical: the node functions, the decisions, the steps and every final state. LangGraph adds a declared schema, a reducer
per key, streamed updates and a built-in step limit (`recursion_limit`). You stop maintaining the runner, and later you get
parallel branches and saved state.

### 5e. Leave the reducer out
Same keys, `notes` as a plain list, same graph, same incident:

In [9]:
plain = build(TriageNoReducer).invoke(start_state("upstream_deploy-0"))
print("with operator.add :", final_graph["notes"])
print("no reducer        :", plain["notes"])

with operator.add : ['alert names checkout', "unhealthy: ['checkout', 'payments']", 'read checkout logs: 4 bad lines', 'errors come from upstream payments', 'read payments logs: 1 bad lines', 'cause = bad_deploy']
no reducer        : ['cause = bad_deploy']


Each node wrote a one-item list, and each write replaced the one before it, with no error and no warning.

## 6. Where this shows up in agents: add_messages

In chapter 3 your ReAct loop appended every reply to a message list by hand. In LangGraph that list becomes a state key
with the `add_messages` reducer, so each node returns only its new messages. Every message has an id: **a new id is
appended, a known id replaces the old message.** Lesson 4.3 builds the ReAct loop on this.

In [10]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

class Chat(TypedDict):
    messages: Annotated[list, add_messages]

def agent(state):
    return {"messages": [AIMessage("I will read the checkout logs.", id="a1")]}

def edit(state):          # same id -> add_messages REPLACES that message
    return {"messages": [AIMessage("I will read the payments logs.", id="a1")]}

def chat_graph(*nodes):
    cg = StateGraph(Chat)
    prev = START
    for fn in nodes:
        cg.add_node(fn.__name__, fn)
        cg.add_edge(prev, fn.__name__)
        prev = fn.__name__
    cg.add_edge(prev, END)
    return cg.compile()

out = chat_graph(agent, edit).invoke({"messages": [HumanMessage("PAGER: checkout error rate above 5%", id="h1")]})
for m in out["messages"]:
    print(f"  {type(m).__name__:12} id={m.id}  {m.content}")
print(len(out["messages"]), "messages, not 3")

  HumanMessage id=h1  PAGER: checkout error rate above 5%
  AIMessage    id=a1  I will read the payments logs.
2 messages, not 3


## 7. Traps
All four fail without an error, so each cell shows the damage.

**Trap 1: a list key with no reducer.** Only the last write survives (section 5e). Evidence has no reducer on purpose; notes needs one.

In [11]:
print("no reducer -> notes:", plain["notes"], "| same as hand runner?", plain == hand)

no reducer -> notes: ['cause = bad_deploy'] | same as hand runner? False


**Trap 2: a node returns the whole notes list.** With `operator.add`, the old notes are appended a second time.

In [12]:
def classify_whole(state):
    up = classify(state)
    return {"cause": up["cause"], "notes": state["notes"] + up["notes"]}   # returns ALL notes

NODES_BAD = {**NODES, "classify": classify_whole}
g = StateGraph(Triage)
for name, fn in NODES_BAD.items():
    g.add_node(name, fn)
g.add_edge(START, "read_alert"); g.add_edge("read_alert", "check_health")
g.add_conditional_edges("check_health", after_health, ["read_logs", "false_alarm"])
g.add_conditional_edges("read_logs", after_logs, ["follow_upstream", "classify"])
g.add_edge("follow_upstream", "read_logs"); g.add_edge("classify", END); g.add_edge("false_alarm", END)
dup = g.compile().invoke(start_state("upstream_deploy-0"))
print(len(dup["notes"]), "notes instead of 6")

11 notes instead of 6


**Trap 3: reusing a message id by accident.** `add_messages` replaces instead of adding: the checkout line from `agent` is gone (section 6).

In [13]:
print([m.content for m in out["messages"] if m.id == "a1"])

['I will read the payments logs.']


**Trap 4: expecting the framework to fix your flow.** Identical final states mean identical mistakes: the graph also misses the 6 stuck consumers. And LangGraph's own step limit replaces your `max_steps`:

In [14]:
from collections import Counter
from langgraph.errors import GraphRecursionError

ok = Counter()
for iid, inc in INCIDENTS.items():
    f = graph.invoke(start_state(iid))
    ok["cause"] += f["cause"] == inc.label["cause"]
    ok["service"] += f["suspect"] == inc.label["cause_service"]
print(f"StateGraph: right service {ok['service']}/60 | right cause {ok['cause']}/60")

try:
    graph.invoke(start_state("upstream_deploy-0"), {"recursion_limit": 4})
except GraphRecursionError as e:
    print("GraphRecursionError:", str(e).splitlines()[0])

StateGraph: right service 54/60 | right cause 54/60
GraphRecursionError: Recursion limit of 4 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


## 8. Exercises
Each exercise has a starter cell (`# YOUR CODE HERE`) and a check cell. No API needed.

### Exercise 1 (the video's exercise): remove the reducer, predict, then run

Without a reducer on `notes`, what are the final notes of `false_alarm-0`? Write your prediction in `predicted`
(a list of strings), then set `actual` to the notes from `build(TriageNoReducer)`.

In [15]:
predicted = []      # YOUR CODE HERE: your guess first
actual = None       # YOUR CODE HERE: then run it

In [16]:
try:
    assert actual is not None, "run build(TriageNoReducer) on false_alarm-0 and keep its notes"
    assert actual == ['all green: false alarm, change nothing'], f"actual looks wrong: {actual}"
    assert predicted == actual, f"your prediction {predicted} differs from the run {actual}"
    print("notes:", actual)
    print("✓ Exercise 1 passed")
except AssertionError as e:
    print("✗ Exercise 1 not yet:", e)

✗ Exercise 1 not yet: run build(TriageNoReducer) on false_alarm-0 and keep its notes


### Exercise 2: give evidence a reducer

Write `TriageEv`: like `Triage`, but `evidence` also uses `operator.add`. Predict `n_evidence`, the number of evidence
lines at the end of `upstream_deploy-0`, then check it. Careful: `after_logs` reads the evidence too.

In [17]:
class TriageEv(TypedDict):
    # YOUR CODE HERE (all eight keys)
    notes: Annotated[list, operator.add]

n_evidence = 0      # YOUR CODE HERE: your prediction

In [18]:
try:
    ev = build(TriageEv).invoke(start_state("upstream_deploy-0"))
    assert "evidence" in TriageEv.__annotations__, "TriageEv needs the evidence key"
    assert len(ev["evidence"]) > 1, "evidence still overwritten: add operator.add to evidence"
    assert n_evidence == len(ev["evidence"]), (f"your prediction {n_evidence} is off: the old checkout lines stay, "
                                               "so what does after_logs see after step 5?")
    print("evidence lines:", len(ev["evidence"]), "| hops:", ev["hops"])
    print("✓ Exercise 2 passed")
except AssertionError as e:
    print("✗ Exercise 2 not yet:", e)
except Exception as e:
    print("✗ Exercise 2 not yet:", type(e).__name__, str(e)[:120])

✗ Exercise 2 not yet: KeyError 'alert'


### Exercise 3: append instead of replace

Write a node `followup` that adds a **new** AI message "Payments logs show a KeyError." so that
`chat_graph(agent, followup)` ends with 3 messages, ids `h1`, `a1`, `a2`.

In [19]:
def followup(state):
    # YOUR CODE HERE
    return {"messages": []}

In [20]:
try:
    o = chat_graph(agent, followup).invoke({"messages": [HumanMessage("PAGER: checkout error rate above 5%", id="h1")]})
    ids = [m.id for m in o["messages"]]
    assert ids == ["h1", "a1", "a2"], f"ids are {ids}"
    assert o["messages"][1].content == "I will read the checkout logs.", "a1 must keep its text"
    assert "KeyError" in o["messages"][2].content
    print("✓ Exercise 3 passed")
except AssertionError as e:
    print("✗ Exercise 3 not yet:", e)

✗ Exercise 3 not yet: ids are ['h1', 'a1']


### Exercise 4: a reducer of your own

Write `keep_last(n)` that returns a reducer keeping only the last `n` items of `old + new`. Use it for `notes`
in `TriageLast3` (notes keeps 3), and run `upstream_deploy-0`.

In [21]:
def keep_last(n):
    # YOUR CODE HERE
    return operator.add

class TriageLast3(TypedDict):
    incident: str
    alert: str
    suspect: str | None
    unhealthy: list
    evidence: list
    hops: int
    cause: str | None
    notes: Annotated[list, keep_last(3)]

In [22]:
try:
    r = keep_last(2)
    assert r([1, 2], [3]) == [2, 3], f"keep_last(2)([1, 2], [3]) gave {r([1, 2], [3])}"
    last3 = build(TriageLast3).invoke(start_state("upstream_deploy-0"))["notes"]
    assert last3 == final_graph["notes"][-3:], f"got {last3}"
    print(last3)
    print("✓ Exercise 4 passed")
except AssertionError as e:
    print("✗ Exercise 4 not yet:", e)

✗ Exercise 4 not yet: keep_last(2)([1, 2], [3]) gave [1, 2, 3]


## Solutions (try first!)

In [23]:
predicted = ['all green: false alarm, change nothing']   # each node overwrites; false_alarm writes last
actual = build(TriageNoReducer).invoke(start_state("false_alarm-0"))["notes"]

In [24]:
class TriageEv(TypedDict):
    incident: str
    alert: str
    suspect: str | None
    unhealthy: list
    evidence: Annotated[list, operator.add]
    hops: int
    cause: str | None
    notes: Annotated[list, operator.add]

n_evidence = 4      # 2 checkout lines (step 3) + 1 payments line (step 5); the checkout "upstream" lines
                    # stay, hops 2 < 3, so the loop runs once more: +1 payments line, then hops 3 -> classify

In [25]:
def followup(state):
    return {"messages": [AIMessage("Payments logs show a KeyError.", id="a2")]}

In [26]:
def keep_last(n):
    def reducer(old, new):
        return (old + new)[-n:]
    return reducer

class TriageLast3(TypedDict):
    incident: str
    alert: str
    suspect: str | None
    unhealthy: list
    evidence: list
    hops: int
    cause: str | None
    notes: Annotated[list, keep_last(3)]